In [ ]:
import rasterio
import numpy as np
import os
from rasterio.enums import Resampling

# --- Localisation des fichiers Landsat ---
base_path = r"C:\Users\scolzy\Stage_SC_26\OMAN_Rasters_test\05_TELEDETECTION\01_LANDSAT"
prefix = "HLS.L30.T40QFJ.2025043T062900.v2.0"

# --- Définition des combinaisons ---
band_combinations = {
    "NaturalColor_432": (4, 3, 2),
    "FalseColor_543": (5, 4, 3),
    "Geological_753": (7, 5, 3),
    "Geological_bis_347": (3, 4, 7),
}

# --- Chargement des bandes Landsat ---
bands = {}
nodata_value = -9999.0

for i in range(1, 8):
    file_path = os.path.join(base_path, f"{prefix}.B0{i}.TIF")
    if os.path.exists(file_path):
        with rasterio.open(file_path) as src:
            band_data = src.read(1, resampling=Resampling.nearest).astype(np.float32)
            band_data[band_data == nodata_value] = 0
            bands[i] = band_data
            profile = src.profile
    else:
        print(f"⚠️ Fichier manquant : {file_path}")

# --- Génération des GeoTIFF ---
for name, (r, g, b) in band_combinations.items():
    if r in bands and g in bands and b in bands:
        rgb_image = np.stack([bands[r], bands[g], bands[b]], axis=0)
        min_val, max_val = np.nanmin(rgb_image), np.nanmax(rgb_image)
        rgb_image = ((rgb_image - min_val) / (max_val - min_val) * 65535).astype(np.uint16)
        profile.update(count=3, dtype=rasterio.uint16, nodata=0)
        output_file = os.path.join(base_path, f"{prefix}_{name}.tif")
        with rasterio.open(output_file, "w", **profile) as dst:
            dst.write(rgb_image)
        print(f"✅ Image {output_file} générée ({r}-{g}-{b})")
    else:
        print(f"❌ Impossible de générer {name}, une ou plusieurs bandes sont manquantes.")

print("🎉 Traitement terminé !")


In [ ]:
import rasterio
import numpy as np
import os
from rasterio.enums import Resampling

base_path = r"C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE"
prefix = "BAN_EMPRISE_SECTEUR_TEST_LANDSAT8"

band_combinations = {
    "NaturalColor_432": (4, 3, 2),      # Couleurs naturelles : Rouge=B4, Vert=B3, Bleu=B2
    "FalseColor_543": (5, 4, 3),        # Fausses couleurs : met en valeur la végétation
    "Geological_753": (7, 5, 3),        # Composition géologique classique
    "Geological_bis_347": (3, 4, 7),    # Variante géologique
}

bands = {}
nodata_value = -9999.0

for i in range(1, 8):
    file_path = os.path.join(base_path, f"{prefix}_B{i}_FUSION_2024_EM.tif")
    if os.path.exists(file_path):
        with rasterio.open(file_path) as src:
            band_data = src.read(1, resampling=Resampling.nearest).astype(np.float32)
            band_data[band_data == nodata_value] = 0
            bands[i] = band_data
            profile = src.profile
    else:
        print(f"⚠️ Fichier manquant : {file_path}")

for name, (r, g, b) in band_combinations.items():
    if r in bands and g in bands and b in bands:
        rgb_image = np.stack([bands[r], bands[g], bands[b]], axis=0)
        min_val, max_val = np.nanmin(rgb_image), np.nanmax(rgb_image)
        rgb_image = ((rgb_image - min_val) / (max_val - min_val) * 65535).astype(np.uint16)
        profile.update(count=3, dtype=rasterio.uint16, nodata=0)
        output_file = os.path.join(base_path, f"{prefix}_{name}.tif")
        with rasterio.open(output_file, "w", **profile) as dst:
            dst.write(rgb_image)
        print(f"✅ Image {output_file} générée ({r}-{g}-{b})")
    else:
        print(f"❌ Impossible de générer {name}, une ou plusieurs bandes sont manquantes.")

print("🎉 Traitement terminé !")


In [ ]:
import rasterio
import numpy as np
import os
from rasterio.enums import Resampling

base_path = r"C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE"
prefix = "BAN_EMPRISE_SECTEUR_TEST_LANDSAT8"

# Format : "nom": (numérateur, dénominateur)
band_ratios = {
    "FerricOxydes_42":     (4, 2),  # B4/B2 : oxydes de fer
    "Clays_hydroxyles_65": (6, 5),  # B6/B5 : minéraux argileux
    "Clays_hydroxyles_75": (7, 5),  # B7/B5 : argiles + alunite
    "IOI_43":              (4, 3),  # B4/B3 : oxyde de fer
    "Si_31":               (3, 1),  # B3/B1 : silice
    "CI_63":               (6, 3),  # B6/B3 : carbonates
    "Ratio_54":            (5, 4),  # B5/B4
    "Ratio_67":            (6, 7),  # B6/B7
}

bands = {}
nodata_value = -9999.0

for i in range(1, 8):
    file_path = os.path.join(base_path, f"{prefix}_B{i}_FUSION_2024_EM.tif")
    if os.path.exists(file_path):
        with rasterio.open(file_path) as src:
            band_data = src.read(1, resampling=Resampling.nearest).astype(np.float32)
            band_data[band_data == nodata_value] = 0
            bands[i] = band_data
            profile = src.profile
    else:
        print(f"⚠️ Fichier manquant : {file_path}")

for name, (num, den) in band_ratios.items():
    if num in bands and den in bands:
        with np.errstate(divide="ignore", invalid="ignore"):  # Supprime les warnings de division par zéro
            ratio = np.where(bands[den] != 0, bands[num] / bands[den], np.nan)  # NaN si dénominateur = 0
        min_val, max_val = np.nanmin(ratio), np.nanmax(ratio)
        ratio = ((ratio - min_val) / (max_val - min_val) * 65535).astype(np.uint16)
        profile_ratio = profile.copy()
        profile_ratio.update(count=1, dtype=rasterio.float32, nodata=np.nan)
        output_file = os.path.join(base_path, f"{prefix}_{name}.tif")
        with rasterio.open(output_file, "w", **profile_ratio) as dst:
            dst.write(ratio.astype(np.float32), 1)
        print(f"✅ Ratio {output_file} généré (B{num}/B{den})")
    else:
        print(f"❌ Ratio {name} impossible, bande manquante.")

print("🎉 Ratios terminés !")


In [ ]:
import rasterio
import numpy as np
import os
from rasterio.enums import Resampling

base_path = r"C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE"
prefix = "BAN_EMPRISE_SECTEUR_TEST_LANDSAT8"

bands = {}
nodata_value = -9999.0

for i in range(1, 8):
    file_path = os.path.join(base_path, f"{prefix}_B{i}_FUSION_2024_EM.tif")
    if os.path.exists(file_path):
        with rasterio.open(file_path) as src:
            band_data = src.read(1, resampling=Resampling.nearest).astype(np.float32)
            band_data[band_data == nodata_value] = 0
            bands[i] = band_data
            profile = src.profile
    else:
        print(f"⚠️ Fichier manquant : {file_path}")

# Calculs des indices complexes (Barnaby 2021)
with np.errstate(divide='ignore', invalid='ignore'):
    ferric_iron_2                    = np.where((bands[5] != 0) & (bands[2] != 0), (bands[4]/bands[2])*(bands[4]+bands[6])/bands[5], np.nan)  # (B4/B2)*(B4+B6)/B5
    ferrous_iron_3p6_4p5             = np.where((bands[4]+bands[5]) != 0, (bands[3]+bands[6])/(bands[4]+bands[5]), np.nan)                     # (B3+B6)/(B4+B5)
    clay_sulfate_mica_marble_6d7_5d4 = np.where((bands[7] != 0) & (bands[4] != 0), bands[6]/bands[7] - bands[5]/bands[4], np.nan)             # B6/B7 - B5/B4
    iron_sulfate_2d1_5d4             = np.where((bands[1] != 0) & (bands[4] != 0), bands[2]/bands[1] - bands[5]/bands[4], np.nan)             # B2/B1 - B5/B4
    hydrated_minerals_5m6_6p5        = np.where((bands[6]+bands[5]) != 0, (bands[5]-bands[6])/(bands[6]+bands[5]), np.nan)                     # (B5-B6)/(B6+B5)
    clay_alteration_minerals_6m7_6p7 = np.where((bands[6]+bands[7]) != 0, (bands[6]-bands[7])/(bands[6]+bands[7]), np.nan)                     # (B6-B7)/(B6+B7)
    litho_discrim_6m2_6p2            = np.where((bands[6]+bands[2]) != 0, (bands[6]-bands[2])/(bands[6]+bands[2]), np.nan)                     # (B6-B2)/(B6+B2)
    alteration_minerals_6m5_6p5      = np.where((bands[6]+bands[5]) != 0, (bands[6]-bands[5])/(bands[6]+bands[5]), np.nan)                     # (B6-B5)/(B6+B5)

complex_indices = {
    'Ferric_Iron_4_2x4p6_5':                    ferric_iron_2,
    'Ferrous_Iron_3p6_4p5':             ferrous_iron_3p6_4p5,
    'Clay_Sulfate_Mica_Marble_6d7_5d4': clay_sulfate_mica_marble_6d7_5d4,
    'Hydrated_Minerals_5m6_6p5':        hydrated_minerals_5m6_6p5,
    'Clay_Alteration_Minerals_6m7_6p7': clay_alteration_minerals_6m7_6p7,
    'Litho_Discrimination_6m2_6p2':     litho_discrim_6m2_6p2,
    'Alteration_Minerals_6m5_6p5':      alteration_minerals_6m5_6p5,
    'Iron_Sulfate_2d1_5d4':             iron_sulfate_2d1_5d4
}

profile_ratio = profile.copy()
profile_ratio.update(count=1, dtype=rasterio.float32, nodata=np.nan)

for name, index in complex_indices.items():
    min_val, max_val = np.nanmin(index), np.nanmax(index)
    index = ((index - min_val) / (max_val - min_val) * 65535).astype(np.uint16)
    output_file = os.path.join(base_path, f"{prefix}_{name}.tif")
    with rasterio.open(output_file, 'w', **profile_ratio) as dst:
        dst.write(index.astype(np.float32), 1)
    print(f"✅ {name} généré → min={np.nanmin(index):.3f}, max={np.nanmax(index):.3f}")

print("🎉 Indices complexes terminés !")


[1, 2, 3, 4, 5, 6, 7]
✅ Ferric_Iron_4_2x4p6_5 généré → min=0.000, max=65535.000
✅ Ferrous_Iron_3p6_4p5 généré → min=0.000, max=65535.000
✅ Clay_Sulfate_Mica_Marble_6d7_5d4 généré → min=0.000, max=65535.000
✅ Hydrated_Minerals_5m6_6p5 généré → min=0.000, max=65535.000
✅ Clay_Alteration_Minerals_6m7_6p7 généré → min=0.000, max=65535.000
✅ Litho_Discrimination_6m2_6p2 généré → min=0.000, max=65535.000
✅ Alteration_Minerals_6m5_6p5 généré → min=0.000, max=65535.000
✅ Iron_Sulfate_2d1_5d4 généré → min=0.000, max=65535.000
🎉 Indices complexes terminés !


In [ ]:
base_path = r"C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE"
prefix = "BAN_EMPRISE_SECTEUR_TEST_LANDSAT8"

with rasterio.open(os.path.join(base_path, f"{prefix}_Clays_hydroxyles_75.tif")) as source:
    print(f"--- Informations pour : {source.name} ---")
    print(f"Dimensions : {source.width} x {source.height}")
    print(f"Nombre de bandes : {source.count}")
    print(f"CRS : {source.crs}")
    print(f"Emprise : {source.bounds}")
    print(f"Résolution : {source.res}")
    print(f"Transform : \n{source.transform}")
    data = source.read(1, masked=True)
    print(f"\n--- Valeurs ---")
    print(f"Min : {data.min():.2f} ")
    print(f"Max : {data.max():.2f} ")
    print(f"Moyenne : {data.mean():.2f} ")
    print(f"Valeur NoData : {source.nodata}")
    print(f"Type de données : {source.dtypes[0]}")


In [ ]:
import rasterio
import numpy as np
import os
from sklearn.decomposition import PCA

base_path = r"C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE"
prefix = "BAN_EMPRISE_SECTEUR_TEST_LANDSAT8"

# --- Chargement des bandes 2 à 7 (on garde celles qui existent) ---
bands_pca = {}
nodata_value = -9999.0
for i in range(2, 8):
    file_path = os.path.join(base_path, f"{prefix}_B{i}_FUSION_2024_EM.tif")
    if os.path.exists(file_path):
        with rasterio.open(file_path) as src:
            b = src.read(1).astype(np.float32)
            b[b == nodata_value] = np.nan  # nodata explicite
            b[b == 0] = np.nan             # pixels nuls = bords/nodata
            bands_pca[i] = b
            profile = src.profile
            shape = b.shape

print(f"Bandes utilisées pour l'ACP : {list(bands_pca.keys())}")

# --- Empilement en matrice (pixels x bandes) ---
stack = np.array([bands_pca[i] for i in sorted(bands_pca.keys())])  # (n_bands, rows, cols)
n_bands, rows, cols = stack.shape
X = stack.reshape(n_bands, -1).T                                    # (n_pixels, n_bands)

# --- Masque : pixels sans NaN dans toutes les bandes ---
valid_mask = ~np.isnan(X).any(axis=1)                               # booléen (n_pixels,)
X_valid = X[valid_mask]                                             # (n_valid_pixels, n_bands)

# --- ACP (echantillonnage pour accelerer le calcul) ---
n_sample = min(500000, X_valid.shape[0])                            # max 500k pixels pour l'entrainement
idx_sample = np.random.choice(X_valid.shape[0], n_sample, replace=False)
pca = PCA()
pca.fit(X_valid[idx_sample])
print(f"Variance expliquée par PC : {pca.explained_variance_ratio_.round(3)}")

X_pca = pca.transform(X_valid)                                      # (n_valid_pixels, n_components)

# --- Reconstruction des images PC en 2D ---
pc_images = []
for pc_idx in range(min(3, n_bands)):                                # PC1, PC2, PC3
    pc_img = np.full(rows * cols, np.nan, dtype=np.float32)
    pc_img[valid_mask] = X_pca[:, pc_idx]
    pc_images.append(pc_img.reshape(rows, cols))

# --- Normalisation percentile 2-98 pour chaque PC ---
rgb_pca = np.zeros((3, rows, cols), dtype=np.uint16)
for idx, pc_img in enumerate(pc_images):
    vmin, vmax = np.nanpercentile(pc_img, 2), np.nanpercentile(pc_img, 98)
    rgb_pca[idx] = np.clip((pc_img - vmin) / (vmax - vmin) * 65535, 0, 65535).astype(np.uint16)

# --- Export GeoTIFF RGB ---
profile_pca = profile.copy()
profile_pca.update(count=3, dtype=rasterio.uint16, nodata=0)

output_file = os.path.join(base_path, f"{prefix}_PCA_PC123.tif")
with rasterio.open(output_file, 'w', **profile_pca) as dst:
    dst.write(rgb_pca)

print(f"✅ Raster ACP généré : {output_file} (PC1=R, PC2=G, PC3=B)")


Bandes utilisées pour l'ACP : [2, 3, 4, 5, 6, 7]
Variance expliquée par PC : [0.901 0.057 0.028 0.011 0.002 0.001]
✅ Raster ACP généré : C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE\BAN_EMPRISE_SECTEUR_TEST_LANDSAT8_PCA_PC123.tif (PC1=R, PC2=G, PC3=B)


In [1]:
import rasterio
import numpy as np
import os
from sklearn.decomposition import PCA

base_path = r"C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE"
prefix = "BAN_EMPRISE_SECTEUR_TEST_LANDSAT8"

# --- 1. Chargement des bandes 2 à 7 ---
bands_pca = {}
nodata_value = -9999.0
for i in range(2, 8):
    file_path = os.path.join(base_path, f"{prefix}_B{i}_FUSION_2024_EM.tif")
    if os.path.exists(file_path):
        with rasterio.open(file_path) as src:
            b = src.read(1).astype(np.float32)
            b[b == nodata_value] = np.nan  # nodata explicite
            b[b == 0] = np.nan             # pixels nuls = bords/nodata
            bands_pca[i] = b
            profile = src.profile
            shape = b.shape

print(f"Bandes utilisées pour l'ACP : {list(bands_pca.keys())}")

print(sorted(bands_pca.keys()))

# --- 2. Empilement en matrice (pixels x bandes) ---
stack = np.array([bands_pca[i] for i in sorted(bands_pca.keys())])  # (n_bands, rows, cols)
n_bands, rows, cols = stack.shape
X = stack.reshape(n_bands, -1).T                                    # (n_pixels, n_bands)

# --- 3. Masque : pixels sans NaN dans toutes les bandes ---
valid_mask = ~np.isnan(X).any(axis=1)                               # booléen (n_pixels,)
X_valid = X[valid_mask]                                             # (n_valid_pixels, n_bands)

# --- 4. ACP (échantillonnage pour accélérer le calcul) ---
n_sample = min(500000, X_valid.shape[0])                            # max 500k pixels pour l'entraînement
idx_sample = np.random.choice(X_valid.shape[0], n_sample, replace=False)
pca = PCA()
pca.fit(X_valid[idx_sample])
print(f"Variance expliquée par PC : {pca.explained_variance_ratio_.round(3)}")

X_pca = pca.transform(X_valid)                                      # (n_valid_pixels, n_components)

# --- 5. Reconstruction des images PC en 2D ---
pc_images = []
for pc_idx in range(min(3, n_bands)):                               # PC1, PC2, PC3
    pc_img = np.full(rows * cols, np.nan, dtype=np.float32)
    pc_img[valid_mask] = X_pca[:, pc_idx]
    pc_images.append(pc_img.reshape(rows, cols))

# --- 6. Normalisation et assignation géologique RGB (R=PC3, G=PC1, B=PC2) ---
rgb_pca = np.zeros((3, rows, cols), dtype=np.uint16)

# Cartographie des indices de pc_images (0=PC1, 1=PC2, 2=PC3) vers le RGB de sortie
# out_idx 0 = Rouge (PC3) | out_idx 1 = Vert (PC1) | out_idx 2 = Bleu (PC2)
rgb_mapping = [2, 0, 1] 

for out_idx, pc_idx in enumerate(rgb_mapping):
    pc_img = pc_images[pc_idx]
    vmin, vmax = np.nanpercentile(pc_img, 2), np.nanpercentile(pc_img, 98)
    
    if vmax - vmin == 0:
        rgb_pca[out_idx] = 0
    else:
        rgb_pca[out_idx] = np.clip((pc_img - vmin) / (vmax - vmin) * 65535, 0, 65535).astype(np.uint16)

# --- 7. Export GeoTIFF RGB ---
profile_pca = profile.copy()
profile_pca.update(count=3, dtype=rasterio.uint16, nodata=0)

output_file = os.path.join(base_path, f"{prefix}_PCA_PC312.tif")
with rasterio.open(output_file, 'w', **profile_pca) as dst:
    dst.write(rgb_pca)

print(f"✅ Raster ACP ajusté généré : {output_file} (PC3=R, PC1=G, PC2=B)")

Bandes utilisées pour l'ACP : [2, 3, 4, 5, 6, 7]
[2, 3, 4, 5, 6, 7]
Variance expliquée par PC : [0.901 0.057 0.028 0.011 0.002 0.001]
✅ Raster ACP ajusté généré : C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE\BAN_EMPRISE_SECTEUR_TEST_LANDSAT8_PCA_PC312.tif (PC3=R, PC1=G, PC2=B)


In [2]:
import re

numeros_cp = [int(x) for x in re.findall(r"\d+","R=CP3, G=CP1, B=CP2" )]

rgb_mapping = [num -1 for num in numeros_cp]
print(rgb_mapping)

[2, 0, 1]


In [ ]:
import rasterio
import numpy as np
import os
from rasterio.enums import Resampling

base_path = r"C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE"
prefix = "BAN_EMPRISE_SECTEUR_TEST_LANDSAT8"

band_combinations = {
    "NaturalColor_432": (4, 3, 2),      # Couleurs naturelles : Rouge=B4, Vert=B3, Bleu=B2
    "FalseColor_543": (5, 4, 3),        # Fausses couleurs : met en valeur la végétation
    "Geological_753": (7, 5, 3),        # Composition géologique classique
    "Geological_bis_347": (3, 4, 7),    # Variante géologique
}

bands = {}
nodata_value = -9999.0

for i in range(1, 8):
    file_path = os.path.join(base_path, f"{prefix}_B{i}_FUSION_2024_EM.tif")
    if os.path.exists(file_path):
        with rasterio.open(file_path) as src:
            band_data = src.read(1, resampling=Resampling.nearest).astype(np.float32)
            band_data[band_data == nodata_value] = 0
            bands[i] = band_data
            profile = src.profile
    else:
        print(f"⚠️ Fichier manquant : {file_path}")

for name, (r, g, b) in band_combinations.items():
    if r in bands and g in bands and b in bands:
        rgb_image = np.stack([bands[r], bands[g], bands[b]], axis=0)
        min_val, max_val = np.nanmin(rgb_image), np.nanmax(rgb_image)
        rgb_image = ((rgb_image - min_val) / (max_val - min_val) * 65535).astype(np.uint16)
        profile.update(count=3, dtype=rasterio.uint16, nodata=0)
        output_file = os.path.join(base_path, f"{prefix}_{name}.tif")
        with rasterio.open(output_file, "w", **profile) as dst:
            dst.write(rgb_image)
        print(f"✅ Image {output_file} générée ({r}-{g}-{b})")
    else:
        print(f"❌ Impossible de générer {name}, une ou plusieurs bandes sont manquantes.")

print("🎉 Traitement terminé !")